# 15.4 - Grade-0 gingival review packet

Organize existing D2 validation evidence for region review. Source grades are retained. The gallery does not certify the clinical labels.

## Load the locked validation records

Reuse the saved predictions at confidence 0.25. Select images with at least one source grade-0 region, and verify each selected image against its checksum.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw
from dentalselfie.detection_audit import box_iou, match_boxes

root = Path.cwd()
while not (root / 'pyproject.toml').exists():
    if root == root.parent:
        raise RuntimeError('Project root not found')
    root = root.parent
report = root / 'docs/reports/gingivitis_binary'
source = pd.read_csv(root / 'data/manifests/gum.splits.csv').query("split == 'val'").set_index('sample_id')
coords = ['x1','y1','x2','y2']
grade0 = pd.read_csv(report / 'audit_grade0_regions.csv').query("model == 'YOLOv8n' and confidence == 0.25")
positive = pd.read_csv(report / 'audit_regions.csv').query("model == 'YOLOv8n' and confidence == 0.25")
predictions = pd.read_csv(report / 'audit_predictions.csv').query('confidence >= 0.25')
assert len(grade0) == 49
assert set(grade0.sample_id) <= set(source.index)
print('Grade-0 boxes:', len(grade0), '| validation images containing grade 0:', grade0.sample_id.nunique())

Grade-0 boxes: 49 | validation images containing grade 0: 23


## Create the review gallery and manifest

Red boxes are unmatched predictions overlapping a grade-0 source box at IoU ≥0.50. Cyan outlines are source grade-0 boxes and green outlines in the first panel are positive source annotations. Each source grade-0 box is numbered. The blank review columns are for a human reviewer.

In [2]:
from IPython.display import display
from html import escape
gallery = report / 'grade0_review'
gallery.mkdir(exist_ok=True)
rows, panels = [], []
for sample, negatives in grade0.groupby('sample_id', sort=True):
    path = root / source.loc[sample, 'image']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == source.loc[sample, 'sha256']
    with Image.open(path) as original:
        im = original.convert('RGB')
    im.thumbnail((560, 520))
    positive_boxes = positive.loc[positive.sample_id == sample, coords].to_numpy()
    negative_boxes = negatives[coords].to_numpy()
    canvas = Image.new('RGB', (3 * im.width, im.height + 25), 'white')
    for panel, model in enumerate(['Source annotations','YOLOv8n','YOLOv8s']):
        layer = im.copy()
        draw = ImageDraw.Draw(layer)
        def outline(box, color, text=None):
            x1,y1,x2,y2 = (int(value * size) for value,size in zip(box,(im.width,im.height,im.width,im.height)))
            draw.rectangle((x1,y1,x2,y2),outline=color,width=2)
            if text:
                draw.text((x1+2,max(0,y1-12)),text,fill=color,stroke_width=1,stroke_fill='black')
        if panel == 0:
            for box in positive_boxes:
                outline(box,'#55dd55')
        else:
            group = predictions[(predictions.sample_id == sample) & (predictions.model == model)]
            boxes = group[coords].to_numpy()
            scores = group.confidence.to_numpy()
            events,_ = match_boxes(positive_boxes, boxes, scores)
            unmatched = [event['prediction_index'] for event in events if event['gt_index'] is None]
            overlaps = box_iou(boxes[unmatched],negative_boxes)
            for index, relevant in zip(unmatched,(overlaps >= 0.50).any(axis=1)):
                if relevant:
                    outline(boxes[index],'#ff4444',f'{scores[index]:.2f}')
        for index,box in enumerate(negative_boxes,1):
            outline(box,'#00eeff',str(index))
        canvas.paste(layer,(panel * im.width,25))
        ImageDraw.Draw(canvas).text((panel * im.width+8,7),model,fill='black')
    for index, box in enumerate(negative_boxes, 1):
        iou = box_iou([box],positive_boxes)
        rows.append((sample,index,*box,float(iou.max()) if iou.size else 0.0))
    filename = sample + '.jpg'
    canvas.save(gallery / filename, quality=90)
    panels.append(f'<figure><figcaption>{escape(sample)}</figcaption><img loading="lazy" src="{filename}" alt="Three panels for {escape(sample)}"></figure>')
review = pd.DataFrame(rows,columns=['sample_id','region_number',*coords,'max_iou_with_positive_source_box'])
review['review_status'] = 'pending'
review['review_notes'] = ''
review_path = gallery / 'review_queue.csv'
if not review_path.exists():
    review.to_csv(review_path,index=False)
assert len(review) == 49 and len(panels) == grade0.sample_id.nunique()
html = '<!doctype html><html lang="en"><meta charset="utf-8"><title>Grade-0 gingival review</title><style>body{font:16px system-ui;max-width:1400px;margin:32px auto;padding:0 16px}figure{margin:30px 0}img{width:100%;height:auto}</style><h1>Grade-0 gingival review</h1><p>Cyan: grade-0 annotation. Green: positive source annotation. Red: unmatched prediction overlapping grade 0 at IoU ≥0.50. Confidence cutoff 0.25.</p>' + ''.join(panels) + '</html>'
(gallery / 'index.html').write_text(html,encoding='utf-8')
print('Grade-0 boxes overlapping a positive annotation at IoU ≥ 0.50:', int((review.max_iou_with_positive_source_box >= 0.50).sum()))
print('Review panels saved:',len(panels))
display(review.head())

Grade-0 boxes overlapping a positive annotation at IoU ≥ 0.50: 0
Review panels saved: 23


,sample_id,region_number,x1,y1,x2,y2,max_iou_with_positive_source_box,review_status,review_notes
0,0978c97063c08bbd880e,1,0.610054,0.638641,0.670924,0.741449,0.000000,pending,
1,11ad2154a5c647d33df0,1,0.416576,0.333062,0.520788,0.474552,0.010748,pending,
2,137f62b565f0671a8d28,1,0.342119,0.655742,0.416304,0.768527,0.021331,pending,
3,137f62b565f0671a8d28,2,0.406929,0.649431,0.482201,0.774838,0.000000,pending,
4,137f62b565f0671a8d28,3,0.480162,0.658998,0.552173,0.781148,0.000000,pending,
